# COL-03 — Pilot Baseline Metrics & Visualization

Transforms baseline and sweep artifacts into verified tabular metrics and relative space/temporal plots.
Provides formal machine-readable handoff artifacts for EXP-04 (ThanhTris).

### Adherence & Boundaries:
- 100% data trace back to exact parquet and json artifacts; no manual numbers.
- WINDOW baseline has strictly NULL DBCV with explicit reason `WINDOW_BASELINE_NOT_DENSITY_BASED`.
- Failed windows (mainshock noise) maintain honest failure records without heuristic assignment.
- Spatial plots are rendered in relative kilometers (dx_km, dy_km) and time (delta_time_hours) vs depth (km).
- Pilot scope: DEVELOPMENT_ONLY, PARTIAL coverage.

In [ ]:
import json
import os
import sys
import tempfile
from pathlib import Path
import pandas as pd

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
assert (root / 'notebooks/baseline_visualization.py').is_file(), 'Mở notebook từ root repo hoặc notebooks/'
sys.path.insert(0, str(root / 'notebooks'))
sys.path.insert(0, str(root / 'tests'))

from experiment_harness import load_feature_bundle, MODE_DEVELOPMENT
from baseline_visualization import (
    extract_baseline_metrics,
    export_metrics_table,
    plot_spatial_and_temporal_views,
    plot_algorithm_comparison_summary,
    create_handoff_manifest,
)
print('COL-03 Visualization & Metrics modules imported successfully.')

## 1. Load Development Pilot Bundle

Loads the verified pilot bundle containing 4-D coordinates and relative physical metrics (dx_km, dy_km, delta_time_hours).

In [ ]:
from test_experiment_harness import create_pilot_fixture

bundle_path = os.environ.get('PILOT_BUNDLE', '')
_temp_dir = None
if bundle_path and Path(bundle_path).is_dir():
    pilot_dir = Path(bundle_path).resolve()
else:
    _temp_dir = tempfile.TemporaryDirectory()
    pilot_dir = Path(_temp_dir.name) / 'bundle'
    pilot_dir.mkdir(parents=True, exist_ok=True)
    create_pilot_fixture(pilot_dir)

bundle = load_feature_bundle(pilot_dir, mode=MODE_DEVELOPMENT)
features = bundle['features_table']
print('Loaded Pilot ID:', bundle['bundle_id'])
print('Candidate count:', features.num_rows)
print('Mainshocks count:', bundle['mainshocks_table'].num_rows)

## 2. Extract Verified Metrics Table (CSV & JSON)

Reads directly from baseline/sweep result artifacts. Computes density metrics where applicable (with explicit null + reason for WINDOW and noise windows).

In [ ]:
output_env = os.environ.get('COL03_OUTPUT_DIR')
if output_env:
    out_dir = Path(output_env).resolve()
else:
    out_dir = (root / 'staging/ml-experiments/col03-metrics-plots').resolve()
out_dir.mkdir(parents=True, exist_ok=True)

results_env = os.environ.get('RESULTS_DIR')
if results_env and Path(results_env).is_dir():
    res_dir = Path(results_env).resolve()
elif (root / 'staging/ml-experiments/col02-sweep').is_dir():
    res_dir = (root / 'staging/ml-experiments/col02-sweep').resolve()
elif (root / 'staging/exp02-colab-rosy179/results').is_dir():
    res_dir = (root / 'staging/exp02-colab-rosy179/results').resolve()
else:
    raise ValueError('No baseline/sweep results found')

metrics_rows = extract_baseline_metrics(res_dir, bundle)
csv_p, json_p = export_metrics_table(metrics_rows, out_dir)
print(f"Metrics exported to:\n  {csv_p}\n  {json_p}")

df_metrics = pd.DataFrame(metrics_rows)
df_metrics[['algorithm_name', 'model_config_version', 'total_candidates', 'sequence_members', 'aftershocks', 'noise_candidates', 'dbcv_score', 'window_status']]

## 3. Generate Spatial & Timeline Depth Visualizations

Plots spatial relative coordinates $(dx, dy)$ in km and temporal profile relative to mainshock ($t=0$).

In [ ]:
exported_plots = []
plot_subdirs = [p for p in res_dir.iterdir() if p.is_dir() and (p / 'memberships.parquet').is_file()]
for sdir in plot_subdirs[:3]:
    mem_p = sdir / 'memberships.parquet'
    algo = sdir.name.upper()
    sp_p, tl_p = plot_spatial_and_temporal_views(bundle, mem_p, algo, out_dir)
    exported_plots.extend([sp_p, tl_p])
    print(f"Generated plots for {algo}: {sp_p.name}, {tl_p.name}")

summary_chart_p = plot_algorithm_comparison_summary(metrics_rows, out_dir)
exported_plots.append(summary_chart_p)
print(f"Generated summary comparison chart: {summary_chart_p.name}")

## 4. Packaging Handoff Manifest for ThanhTris (EXP-04)

Generates `handoff_manifest.json` locking all exported tables and plots with SHA-256 checksums.

In [ ]:
all_files = [csv_p, json_p] + exported_plots
manifest_p = create_handoff_manifest(
    output_dir=out_dir,
    pilot_id=bundle['bundle_id'],
    exported_files=all_files,
    recipient='ThanhTris',
)
print('Handoff Manifest generated at:', manifest_p)
manifest_data = json.loads(manifest_p.read_text(encoding='utf-8'))
print(f"Total locked files: {manifest_data['total_files']}")

if _temp_dir is not None:
    _temp_dir.cleanup()
print('All COL-03 artifacts preserved at:', out_dir)